# 08 · Metrics & Threshold Analysis

**Amaç:** Modeller × metrikler matrisini çıkarmak; threshold politikasını iki perspektifle (percentile + alerts/day) raporlamak; demografi-free ablation karşılaştırması yapmak.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

from src.data.loader import load_validated

import json
from pathlib import Path
res = json.loads(Path('../artifacts/eval/all_models.json').read_text())

## Full karşılaştırma tablosu

In [ ]:
rows = []
for name, m in res['models'].items():
    base = name.split('__')
    mname, variant = base[0], base[1] if len(base)>1 else ''
    t = m['test']['core']
    p1 = next((x for x in m['test']['top_k_percentile'] if abs(x['k_frac']-0.01)<1e-9), None)
    p005 = next((x for x in m['test']['top_k_percentile'] if abs(x['k_frac']-0.005)<1e-9), None)
    rows.append({'model': mname, 'variant': variant, 'pr_auc': round(t['pr_auc'],4), 'roc_auc': round(t['roc_auc'],4),
                 'precision@1%': round(p1['precision'],4) if p1 else None,
                 'recall@1%': round(p1['recall'],4) if p1 else None,
                 'recall@0.5%': round(p005['recall'],4) if p005 else None})
cmp = pd.DataFrame(rows).sort_values(['pr_auc','recall@1%'], ascending=False)
cmp

## Demografi ablation

In [ ]:
piv = cmp.pivot_table(index='model', columns='variant', values=['pr_auc','recall@1%','precision@1%'])
piv

## Alerts/day perspektifi (production scenario)

In [ ]:
rows2 = []
for name, m in res['models'].items():
    for s in m['test']['by_alerts_per_day']:
        rows2.append({'model': name, 'alerts_per_day': s['alerts_per_day_target'],
                      'n_alerts_total': s['n_alerts'], 'precision': round(s['precision'],4),
                      'recall': round(s['recall'],4)})
sc = pd.DataFrame(rows2)
sc.pivot_table(index='model', columns='alerts_per_day', values='recall').round(3)

## Sonuç
- En yüksek PR-AUC: tabloda görülüyor.
- Demografi-free vs full PR-AUC farkı küçükse demografi-free model production önerilir.
- Threshold raporlaması iki perspektifte sunuldu: percentile (dataset-relative) ve günlük alert kapasitesi (business scenario).